In [ ]:
!pip install -q "typing_extensions>=4.10.0" --upgrade
!pip install -q duckdb==1.4.4
!pip install -q dbt-duckdb
import sys
sys.exit(0)

In [ ]:
import os
import notebookutils

# -- Resolve paths --
workspace_id = notebookutils.runtime.context.get('currentWorkspaceId')
lakehouse_id = notebookutils.lakehouse.get('DEV_FilmProd_LH2').get('id')

onelake_root = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}'

# DuckLake auto-creates metadata.db on first attach; raw tables read via delta_scan()
os.environ['METADATA_LOCAL_PATH'] = '/lakehouse/default/Files/metadata.db'
os.environ['ROOT_PATH'] = onelake_root

print(f'Workspace: {workspace_id}')
print(f'Lakehouse: {lakehouse_id}')
print(f'OneLake:   {onelake_root}')
print(f'Metadata:  {os.environ["METADATA_LOCAL_PATH"]}')

# -- Run dbt --
os.chdir('/lakehouse/default/Files/dbt')

from dbt.cli.main import dbtRunner
dbt = dbtRunner()

print('\n=== dbt run ===')
res_run = dbt.invoke(['run', '--target', 'fabric', '--profiles-dir', '.'])
print(f'dbt run success: {res_run.success}')

print('\n=== dbt test ===')
res_test = dbt.invoke(['test', '--target', 'fabric', '--profiles-dir', '.'])
print(f'dbt test success: {res_test.success}')

if not res_run.success or not res_test.success:
    raise RuntimeError('dbt failed -- check output above')

print('\nDone.')